# 📐 Notebook 1: UML Basics — what the diagrams mean

## 🛠️ Setup

```bash
cd 07-object-oriented-design/uml-basics
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What is UML?

**UML (Unified Modeling Language)** is a visual vocabulary for describing software. Like
blueprints for a house: before writing code, you draw *what is there* and *how it talks*.

You don't need fancy tools — a whiteboard or ASCII is enough. UML is a convention, not a religion.

### The 3 diagrams you'll use 95% of the time
1. **Class diagram** — *structure*: what objects exist, what they own, how they relate.
2. **Sequence diagram** — *interaction over time*: who calls whom, in what order.
3. **Activity diagram** — *control flow*: the steps a process takes (like a flowchart).

There are ~13 UML diagram types total, but these three cover most interview and real-world needs.


## Class diagrams — the vocabulary

```
┌──────────────────┐
│   ClassName      │  ← the box is the class
├──────────────────┤
│ - privateField   │  ← `-` private, `+` public, `#` protected
│ + publicField    │
├──────────────────┤
│ + method(): Type │  ← signature with return type
└──────────────────┘
```

### Relationships between classes

| Arrow | Meaning | Example |
|---|---|---|
| `──▷`  | **Inheritance** (is-a) | `Dog ──▷ Animal` |
| `──◆`  | **Composition** (strong has-a, lifetime tied) | `House ──◆ Room` |
| `──◇`  | **Aggregation** (weak has-a) | `Team ──◇ Player` |
| `───`  | **Association** (uses / references) | `Order ─── Customer` |
| `┄┄▷`  | **Dependency** (temporarily uses) | `OrderService ┄┄▷ Logger` |

**Multiplicity** goes on the ends: `1`, `0..1`, `*` (many), `1..*`.


## A worked example — library

```
┌─────────┐ 1    *  ┌──────┐ *    1 ┌────────┐
│ Library │◆───────│ Book │────────│ Author │
└─────────┘         └──────┘        └────────┘
                      △
                      │ inheritance
         ┌────────────┼────────────┐
         │            │            │
    ┌────────┐  ┌──────────┐  ┌─────────┐
    │ Ebook  │  │ Audiobook│  │PrintBook│
    └────────┘  └──────────┘  └─────────┘
```

Read it as:
- A Library **is composed of** many Books.
- Each Book **has one** Author, an Author **has many** Books.
- Ebook, Audiobook, PrintBook **inherit from** Book.


## Sequence diagram — who calls whom

Time flows **downward**. Each vertical line is an object. Arrows are messages.

```
User        WebApp        OrderSvc      PaymentSvc
 │             │              │              │
 │──checkout──▶│              │              │
 │             │──createOrder▶│              │
 │             │              │──charge─────▶│
 │             │              │◀─── ok ──────│
 │             │◀── order#42 ─│              │
 │◀── 200 OK ──│              │              │
```

Great for: API flows, auth handshakes, distributed transactions, debugging "who did what, when".


## Activity diagram — the process flowchart

```
  (start)
     │
     ▼
  ┌──────────────┐
  │ add to cart  │
  └──────────────┘
     │
     ▼
  ◇ logged in?  ── no ──▶ ┌────────┐
     │ yes                │  login │
     ▼                    └────────┘
  ┌──────────┐                │
  │ checkout │◀───────────────┘
  └──────────┘
     │
     ▼
   (end)
```

Diamonds = decisions, rectangles = actions. Use it when explaining a *process*, not a system.


## Use case diagram — who uses the system, to do what

A **use case diagram** zooms all the way out. It shows:
- **Actors** (stick figures) — people or other systems that interact with yours.
- **Use cases** (ovals) — goals an actor can achieve with the system.
- **System boundary** (rectangle) — what is inside vs outside your software.

```
                  ┌──────────────────── Online Bookstore ────────────────────┐
                  │                                                          │
       O          │    ( Browse catalog )                                    │
      /|\ ────────┤                                                          │
      / \         │    ( Track delivery )                                    │
    Customer      │                                                          │
       │          │    ( Place order ) ◀┄┄┄┄ «extend» ┄┄┄┄ ( Apply coupon )  │
       └──────────┤           │                                              │
                  │       «include»                                          │            O
                  │           │                                              │           /|\
                  │           ▼                                              │           / \
                  │    ( Charge card ) ─────────────────────────────────────▶├───────  Payment
                  │                                                          │          Gateway
       O          │                                                          │
      /|\ ────────┤    ( Ship package )                                      │
      / \         │                                                          │
    Courier       └──────────────────────────────────────────────────────────┘
```

Read it as:
- **Customer** can browse, track a delivery, and place an order.
- **Placing an order always charges a card** — a solid `«include»`, no branch.
- **Applying a coupon sometimes happens during an order** — a dotted `«extend»`,
  and the arrow points *at* the use case being extended.
- **Payment Gateway** and **Courier** are *actors too*: external systems the
  bookstore talks to but does not own. Drawing them outside the box is the
  point — it's a map of your boundaries.

The two relationships, stated plainly:

| Relationship | Meaning | Direction of the arrow |
|---|---|---|
| `«include»` | A **always** uses B | A ──▶ B (the included one) |
| `«extend»`  | B **sometimes** adds to A | B ┄┄▶ A (the extended one) |

> ⚠️ The arrow directions are the part people get backwards. `«include»` points
> *away* from the base use case; `«extend»` points *towards* it. Mnemonic: the
> optional behaviour is the one reaching in.

Great for: kickoff meetings, scoping an MVP, aligning engineers with product/stakeholders.
Avoid for: implementation detail — use class or sequence diagrams instead.

## State diagram — the life of one object

When a single object has **modes** (e.g., an `Order` is *Pending* → *Paid* → *Shipped*), a state diagram is clearer than a class diagram.

```
   (●) ─────────────▶ [ Pending ] ──pay()───▶ [ Paid ]
                          │                    │
                          cancel()             ship()
                          │                    ▼
                          ▼                [ Shipped ]
                      [ Cancelled ]              │
                          │                    deliver()
                          ▼                    ▼
                         (◉)               [ Delivered ] ──▶ (◉)
```

Use when invalid transitions are a real bug source (payments, workflows, protocols).


## Stereotypes & visibility cheat-sheet

UML lets you tag a class with a **stereotype** in guillemets to clarify its role:

| Stereotype | Meaning | Python equivalent |
|---|---|---|
| `<<abstract>>` | cannot be instantiated on its own | `ABC` + `@abstractmethod` |
| `<<interface>>` | only declares behavior | `Protocol` / `ABC` with only abstract methods |
| `<<enum>>` | fixed set of values | `enum.Enum` |
| `<<service>>` | stateless operation holder | module of functions or a class with no fields |

**Visibility markers** on fields/methods:

| UML | Meaning | Python convention |
|---|---|---|
| `+ name` | public | `name` |
| `- name` | private | `_name` (single underscore = "internal") |
| `# name` | protected | `_name` (Python has no real `protected`) |
| `~ name` | package-private | no direct equivalent |

Python doesn't enforce visibility — the underscore is a **promise**, not a lock. UML is still
useful for *communicating intent*.


## Rule of thumb

> **Draw just enough UML to answer the question you have.**

- Explaining what a system *is* → class diagram.
- Explaining how a request *flows* → sequence diagram.
- Explaining a user's *journey* → activity diagram.

Avoid drawing every getter/setter. Keep diagrams at the level that makes the reader say "ah, I get it."


## Appendix — the same diagrams in Mermaid

ASCII is unbeatable on a whiteboard, but for a README, a PR, or a design doc you
usually want something that renders. [Mermaid](https://mermaid.js.org) is text you
commit next to the code, and GitHub, GitLab, Notion and most wikis render it inline.
Same information, zero drawing tools.

**Class diagram** (the library from earlier):

```
classDiagram
    class Book {
        <<abstract>>
        +title: str
        +author: Author
        +media_type() str
    }
    Library "1" *-- "*" Book : composition
    Book "*" --> "1" Author : association
    Book <|-- Ebook
    Book <|-- Audiobook
    Book <|-- PrintBook
```

Note how the arrow glyphs map one-to-one onto the table at the top of this notebook:
`*--` is composition (◆), `o--` is aggregation (◇), `<|--` is inheritance (▷),
`-->` is association, and `..>` is dependency.

**Sequence diagram** (the checkout flow):

```
sequenceDiagram
    User->>WebApp: checkout()
    WebApp->>OrderSvc: createOrder()
    OrderSvc->>PaymentSvc: charge()
    PaymentSvc-->>OrderSvc: ok
    OrderSvc-->>WebApp: order#42
    WebApp-->>User: 200 OK
```

**State diagram** (the order lifecycle):

```
stateDiagram-v2
    [*] --> Pending
    Pending --> Paid: pay()
    Pending --> Cancelled: cancel()
    Paid --> Shipped: ship()
    Shipped --> Delivered: deliver()
    Delivered --> [*]
    Cancelled --> [*]
```

> 💡 Change the fence above from ` ``` ` to ` ```mermaid ` in any Markdown file that
> supports it and the picture appears. We left them as plain fences here because the
> classic Jupyter Markdown renderer doesn't ship a Mermaid engine — the ASCII versions
> earlier in this notebook are the ones that always render, everywhere.